In [2]:
import pandas as pd
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"

if not os.path.exists("household_power_consumption.txt"):
    urllib.request.urlretrieve(url, "household_power_consumption.zip")
    with zipfile.ZipFile("household_power_consumption.zip", 'r') as z:
        z.extractall(".")

df = pd.read_csv(
    "household_power_consumption.txt",
    sep=';',
    parse_dates={'datetime': ['Date', 'Time']},
    dayfirst=True,
    na_values=['?'],
    infer_datetime_format=True,
    low_memory=False
)

print(df.shape)
print(df.head())

C:\Users\Admin\AppData\Local\Temp\ipykernel_5688\396848744.py:13: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv(
C:\Users\Admin\AppData\Local\Temp\ipykernel_5688\396848744.py:13: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df = pd.read_csv(


(2075259, 8)
             datetime  Global_active_power  Global_reactive_power  Voltage  \
0 2006-12-16 17:24:00                4.216                  0.418   234.84   
1 2006-12-16 17:25:00                5.360                  0.436   233.63   
2 2006-12-16 17:26:00                5.374                  0.498   233.29   
3 2006-12-16 17:27:00                5.388                  0.502   233.74   
4 2006-12-16 17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
0              18.4             0.0             1.0            17.0  
1              23.0             0.0             1.0            16.0  
2              23.0             0.0             2.0            17.0  
3              23.0             0.0             1.0            17.0  
4              15.8             0.0             1.0            17.0  


In [3]:
import timeit

def clean_data(df):
    df = df.dropna()
    numeric_cols = [
        'Global_active_power', 'Global_reactive_power', 'Voltage',
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna()
    return df

t = timeit.Timer(lambda: clean_data(df))
print(f"Час очищення: {t.timeit(1):.4f} сек")
df = clean_data(df)
print(df.shape)

C:\Users\Admin\AppData\Local\Temp\ipykernel_5688\890881799.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors='coerce')


Час очищення: 0.3983 сек
(2049280, 8)


Обрати всі записи, у яких загальна активна споживана потужність 
перевищує 5 кВт. 

In [4]:
def sample_high_power(df):
    return df[df['Global_active_power'] > 5]

t = timeit.Timer(lambda: sample_high_power(df))
print(f"Час: {t.timeit(1):.4f} сек")
s1 = sample_high_power(df)
print(f"Кількість записів: {len(s1)}")
print(s1.head())

Час: 0.0158 сек
Кількість записів: 17547
              datetime  Global_active_power  Global_reactive_power  Voltage  \
1  2006-12-16 17:25:00                5.360                  0.436   233.63   
2  2006-12-16 17:26:00                5.374                  0.498   233.29   
3  2006-12-16 17:27:00                5.388                  0.502   233.74   
11 2006-12-16 17:35:00                5.412                  0.470   232.78   
12 2006-12-16 17:36:00                5.224                  0.478   232.99   

    Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
1               23.0             0.0             1.0            16.0  
2               23.0             0.0             2.0            17.0  
3               23.0             0.0             1.0            17.0  
11              23.2             0.0             1.0            17.0  
12              22.4             0.0             1.0            16.0  


Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них 
виявити ті, у яких пральна машина та холодильних споживають більше, 
ніж бойлер та кондиціонер.

In [5]:
def sample_washer_fridge(df):
    mask = (
        (df['Global_intensity'] >= 19) &
        (df['Global_intensity'] <= 20) &
        (df['Sub_metering_2'] > df['Sub_metering_3'])
    )
    return df[mask]

t = timeit.Timer(lambda: sample_washer_fridge(df))
print(f"Час: {t.timeit(1):.4f} сек")
s2 = sample_washer_fridge(df)
print(f"Кількість записів: {len(s2)}")
print(s2.head())

Час: 0.1012 сек
Кількість записів: 2509
               datetime  Global_active_power  Global_reactive_power  Voltage  \
45  2006-12-16 18:09:00                4.464                  0.136   234.66   
460 2006-12-17 01:04:00                4.582                  0.258   238.08   
464 2006-12-17 01:08:00                4.618                  0.104   239.61   
475 2006-12-17 01:19:00                4.636                  0.140   237.37   
476 2006-12-17 01:20:00                4.634                  0.152   237.17   

     Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
45               19.0             0.0            37.0            16.0  
460              19.6             0.0            13.0             0.0  
464              19.6             0.0            27.0             0.0  
475              19.4             0.0            36.0             0.0  
476              19.4             0.0            35.0             0.0  


Обрати випадковим чином 500000 записів (без повторів елементів 
вибірки), для них обчислити середні величини усіх 3-х груп споживання 
електричної енергії 

In [6]:
def sample_random_500k(df):
    n = min(500_000, len(df))
    sample = df.sample(n=n, replace=False, random_state=42)
    means = sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return sample, means

t = timeit.Timer(lambda: sample_random_500k(df))
print(f"Час: {t.timeit(1):.4f} сек")
s3, means = sample_random_500k(df)
print(f"Кількість записів: {len(s3)}")
print("Середні значення груп споживання:")
print(means)

Час: 0.4132 сек
Кількість записів: 500000
Середні значення груп споживання:
Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
dtype: float64


Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в 
середньому, серед відібраних визначити ті, у яких основне споживання 
електроенергії у вказаний проміжок часу припадає на пральну машину, 
сушарку, холодильник та освітлення (група 2 є найбільшою), а потім 
обрати кожен третій результат із першої половини та кожен четвертий 
результат із другої половини. 

In [7]:
def sample_evening_6kw(df):
    filtered = df[
        (df['datetime'].dt.hour >= 18) &
        (df['Global_active_power'] > 6) &
        (df['Sub_metering_2'] >= df['Sub_metering_1']) &
        (df['Sub_metering_2'] >= df['Sub_metering_3'])
    ]
    mid = len(filtered) // 2
    first_half  = filtered.iloc[:mid].iloc[2::3]   # кожен 3-й
    second_half = filtered.iloc[mid:].iloc[3::4]   # кожен 4-й
    return pd.concat([first_half, second_half])

t = timeit.Timer(lambda: sample_evening_6kw(df))
print(f"Час: {t.timeit(1):.4f} сек")
s4 = sample_evening_6kw(df)
print(f"Кількість записів: {len(s4)}")
print(s4.head())

Час: 0.3872 сек
Кількість записів: 348
                 datetime  Global_active_power  Global_reactive_power  \
43    2006-12-16 18:07:00                6.474                  0.144   
3007  2006-12-18 19:31:00                6.158                  0.442   
17497 2006-12-28 21:01:00                7.062                  0.270   
17500 2006-12-28 21:04:00                7.376                  0.238   
17503 2006-12-28 21:07:00                7.248                  0.000   

       Voltage  Global_intensity  Sub_metering_1  Sub_metering_2  \
43      231.85              27.8             0.0            37.0   
3007    229.08              27.0             0.0            36.0   
17497   235.76              30.2             2.0            65.0   
17500   234.67              31.4             1.0            72.0   
17503   235.34              30.8             1.0            72.0   

       Sub_metering_3  
43               16.0  
3007              0.0  
17497            17.0  
17500            

Пронормувати та стандартизувати вибраний датасет

In [8]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

def normalize_standardize(df):
    cols = [
        'Global_active_power', 'Global_reactive_power', 'Voltage',
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]
    df = df.copy()
    df[[c + '_norm' for c in cols]] = MinMaxScaler().fit_transform(df[cols])
    df[[c + '_std'  for c in cols]] = StandardScaler().fit_transform(df[cols])
    return df

t = timeit.Timer(lambda: normalize_standardize(s4))
print(f"Час: {t.timeit(1):.4f} сек")
df_norm = normalize_standardize(s4)
print(df_norm[['Global_active_power', 'Global_active_power_norm', 'Global_active_power_std']].head())

Час: 0.0539 сек
       Global_active_power  Global_active_power_norm  Global_active_power_std
43                   6.474                  0.100729                -0.538757
3007                 6.158                  0.033005                -0.946479
17497                7.062                  0.226747                 0.219915
17500                7.376                  0.294042                 0.625056
17503                7.248                  0.266610                 0.459903


Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів. 

In [9]:
from scipy import stats

def compute_correlations(df):
    col1, col2 = 'Global_active_power', 'Global_intensity'
    pearson_r,  pearson_p  = stats.pearsonr(df[col1], df[col2])
    spearman_r, spearman_p = stats.spearmanr(df[col1], df[col2])
    print(f"Пірсон:  r={pearson_r:.4f},  p={pearson_p:.4e}")
    print(f"Спірмен: r={spearman_r:.4f}, p={spearman_p:.4e}")

t = timeit.Timer(lambda: compute_correlations(s4))
print(f"Час: {t.timeit(1):.4f} сек")
compute_correlations(s4)

Пірсон:  r=0.9939,  p=0.0000e+00
Спірмен: r=0.9873, p=2.6658e-278
Час: 0.1647 сек
Пірсон:  r=0.9939,  p=0.0000e+00
Спірмен: r=0.9873, p=2.6658e-278


Провести One Hot Encoding категоріального атрибута.

In [10]:
def one_hot_encode(df):
    df = df.copy()
    def get_category(hour):
        if   6  <= hour < 12: return 'morning'
        elif 12 <= hour < 18: return 'afternoon'
        elif 18 <= hour < 22: return 'evening'
        else:                 return 'night'

    df['time_category'] = df['datetime'].dt.hour.apply(get_category)
    print(df['time_category'].value_counts())
    return pd.get_dummies(df, columns=['time_category'], prefix='time')

t = timeit.Timer(lambda: one_hot_encode(df_norm))
print(f"Час: {t.timeit(1):.4f} сек")
df_final = one_hot_encode(df_norm)
print(df_final.shape)
print([c for c in df_final.columns if c.startswith('time_')])

time_category
evening    328
night       20
Name: count, dtype: int64
Час: 0.0380 сек
time_category
evening    328
night       20
Name: count, dtype: int64
(348, 24)
['time_evening', 'time_night']
